# **FINE-TUNING FOR REAL-TIME GLAUCOMA SCREENING**

By **GROUP 129** :
**Prabhmeet Singh** 2022UCM2305 ,
**Aryan Jain** 2022UCM2330 ,
**Radhacharan** 2022UCM2365

Under the Supervision
of
**Dr. Surendra Nagar**

This notebook outlines a machine learning pipeline for automated glaucoma detection from fundus images. The primary goal is to develop a robust and accurate classifier to assist in the early diagnosis of glaucoma.

### Overview:

*   **Model Architecture**: The pipeline utilizes a **EfficientNetB0** backbone for feature extraction, enhanced with a **CBAM (Convolutional Block Attention Module)**, and a custom classification head.
*   **Training Methodology**: A two-phase training approach is employed, starting with a frozen backbone and proceeding to fine-tune the entire model.
*   **Key Techniques**: Important techniques include advanced data augmentation, a custom **Focal Loss** implementation for imbalanced datasets, label smoothing, mixed precision training, and `AdamW` optimization with `CosineDecay` learning rate schedules.

**ENVIRONMENT SETUP**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)

import os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# --- GPU MEMORY GROWTH (prevents TF from grabbing all VRAM) ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# --- MIXED PRECISION (halves memory, enables T4 Tensor Cores) ---
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# --- REPRODUCIBILITY ---
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"Mixed Precision Policy: {tf.keras.mixed_precision.global_policy().name}")

Mounted at /content/drive
TensorFlow version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed Precision Policy: mixed_float16


**CONFIGURATION**

In [3]:
CONFIG = {
    # Model architecture
    "input_size": (224, 224, 3),
    "backbone": "efficientnet-b0",

    # Phase 1: Frozen backbone (feature learning)
    "phase1_batch_size": 64,
    "phase1_lr": 3e-4,
    "phase1_epochs": 12,

    # Phase 2: Fine-tuning
    "phase2_batch_size": 32,
    "phase2_lr": 1e-5,
    "phase2_epochs": 25,

    # Optimizer
    "weight_decay": 1e-5,

    # Focal Loss
    "focal_alpha": 0.65,
    "focal_gamma": 2.0,
    "label_smoothing": 0.05,

    # CBAM
    "cbam_reduction": 16,

    # Data paths
    "base_dir": "/content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus",
    "train_dir": "/content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus/train",
    "val_dir": "/content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus/val",
    "test_dir": "/content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus/test",

    # Save directory
    "save_dir": "/content/drive/MyDrive/Glumoma2.0/GlumomaDataset/models/NineMay",
}

USE_CLASS_WEIGHTS = False
os.makedirs(CONFIG["save_dir"], exist_ok=True)

CLASS_WEIGHT = {
    0: 1.0,   # normal
    1: 1.25   # glaucoma
}

RUN_ID = "9_may_glaucoma"
run_name = f"glaucoma_b0_{RUN_ID}"

print("✓ Configuration ready.")
print(f"  Input: {CONFIG['input_size']}")
print(f"  Backbone: {CONFIG['backbone']}")
print(f"  Phase 1: bs={CONFIG['phase1_batch_size']}, lr={CONFIG['phase1_lr']}, epochs={CONFIG['phase1_epochs']}")
print(f"  Phase 2: bs={CONFIG['phase2_batch_size']}, lr={CONFIG['phase2_lr']}, epochs={CONFIG['phase2_epochs']}")
print(f"  Run: {run_name}")

✓ Configuration ready.
  Input: (224, 224, 3)
  Backbone: efficientnet-b0
  Phase 1: bs=64, lr=0.0003, epochs=12
  Phase 2: bs=32, lr=1e-05, epochs=25
  Run: glaucoma_b0_9_may_glaucoma


**TRAIN AND VALIDATION DATASET COPY TO LOCAL DRIVE FOR FASTER IO**

In [ ]:
import shutil
import os

LOCAL_DATA_DIR = '/content/local_data'
DRIVE_BASE = '/content/drive/MyDrive/Glumoma2.0/GlumomaDataset'

def copy_to_local(src, dst):
    if not os.path.exists(dst):
        print(f"Copying {src} → {dst} ...")
        shutil.copytree(src, dst)
        print("Done.")
    else:
        print(f"Path {dst} already exists.")

copy_to_local(f'{DRIVE_BASE}/full-fundus/train', f'{LOCAL_DATA_DIR}/full-fundus/train')
copy_to_local(f'{DRIVE_BASE}/full-fundus/val',   f'{LOCAL_DATA_DIR}/full-fundus/val')

Copying /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus/train → /content/local_data/full-fundus/train ...
Done.
Copying /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/full-fundus/val → /content/local_data/full-fundus/val ...
Done.


In [ ]:
CONFIG['train_dir'] = f'{LOCAL_DATA_DIR}/full-fundus/train'
CONFIG['val_dir']   = f'{LOCAL_DATA_DIR}/full-fundus/val'

**LABEL ALIASES**

In [ ]:
POSITIVE_CLASS_ALIASES = {"glaucoma", "positive", "abnormal"}
NEGATIVE_CLASS_ALIASES = {"normal", "negative"}

def _normalize_class_name(name):
    return name.strip().lower().replace(" ", "_").replace("-", "_")

def resolve_binary_class_indices(class_names):
    normalized = [_normalize_class_name(c) for c in class_names]

    pos_hits = [i for i, c in enumerate(normalized) if c in POSITIVE_CLASS_ALIASES]
    neg_hits = [i for i, c in enumerate(normalized) if c in NEGATIVE_CLASS_ALIASES]

    if len(pos_hits) != 1:
        raise ValueError(f"Could not resolve exactly one positive class from: {class_names}")
    if len(neg_hits) != 1:
        raise ValueError(f"Could not resolve exactly one negative class from: {class_names}")

    return pos_hits[0], neg_hits[0]

**DATA LOADING**

In [ ]:
import tensorflow as tf

print("Loading datasets from Drive...")

IMG_SIZE = CONFIG["input_size"][:2]  # (224, 224)

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CONFIG["train_dir"],
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=None,
    color_mode="rgb",
    shuffle=True,
    seed=42
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CONFIG["val_dir"],
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=None,
    color_mode="rgb",
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CONFIG["test_dir"],
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=None,
    color_mode="rgb",
    shuffle=False
)

class_names = list(train_ds_raw.class_names)
positive_class_idx, negative_class_idx = resolve_binary_class_indices(class_names)

POSITIVE_CLASS_NAME = class_names[positive_class_idx]
NEGATIVE_CLASS_NAME = class_names[negative_class_idx]

print(f"\nClasses: {class_names}")
print(f"Positive class: {POSITIVE_CLASS_NAME} -> 1")
print(f"Negative class: {NEGATIVE_CLASS_NAME} -> 0")
print("\n✓ Dataset loaded.")

Loading datasets from Drive...
Found 8865 files belonging to 2 classes.
Found 985 files belonging to 2 classes.
Found 2463 files belonging to 2 classes.

Classes: ['glaucoma', 'normal']
Positive class: glaucoma -> 1
Negative class: normal -> 0

✓ Dataset loaded.


**AUGMENTATION & DATA PIPELINE**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

rotation_factor = 10 / 360.0

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=rotation_factor),
    layers.RandomBrightness(factor=0.08),
    layers.RandomContrast(factor=0.15),
], name="augmentation_pipeline")

def remap_to_binary(image, label):
    # Force: positive class = 1, negative class = 0
    label = tf.cast(label, tf.int32)
    label = tf.cast(tf.equal(label, positive_class_idx), tf.float32)
    return image, tf.expand_dims(label, axis=-1)

def process_train(image, label):
    image = data_augmentation(image, training=True)
    return image, label

def build_train_pipeline(ds_raw, batch_size):
    return (
        ds_raw
        .map(remap_to_binary, num_parallel_calls=AUTOTUNE)
        .shuffle(2000, seed=42, reshuffle_each_iteration=True)
        .batch(batch_size, drop_remainder=False)
        .map(process_train, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )

def build_eval_pipeline(ds_raw, batch_size):
    return (
        ds_raw
        .map(remap_to_binary, num_parallel_calls=AUTOTUNE)
        .batch(batch_size, drop_remainder=False)
        .prefetch(AUTOTUNE)
    )

print("✓ Data pipeline builders ready.")

✓ Data pipeline builders ready.


**CBAM ATTENTION MODULE**

In [4]:
class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.dense1 = None
        self.dense2 = None
        self.channels = None
        self.reduced_channels = None

    def build(self, input_shape):
        self.channels = int(input_shape[-1])
        self.reduced_channels = max(self.channels // self.reduction_ratio, 1)

        self.dense1 = layers.Dense(
            self.reduced_channels,
            activation="relu",
            kernel_initializer="he_normal",
            use_bias=True
        )
        self.dense2 = layers.Dense(
            self.channels,
            kernel_initializer="he_normal",
            use_bias=True
        )
        super().build(input_shape)

    def call(self, inputs):
        x = tf.cast(inputs, tf.float32)

        avg_pool = tf.reduce_mean(x, axis=[1, 2])
        max_pool = tf.reduce_max(x, axis=[1, 2])

        avg_out = self.dense2(self.dense1(avg_pool))
        max_out = self.dense2(self.dense1(max_pool))

        attention = tf.nn.sigmoid(avg_out + max_out)
        attention = tf.reshape(attention, [-1, 1, 1, self.channels])

        attention = tf.cast(attention, inputs.dtype)
        return inputs * attention

    def get_config(self):
        config = super().get_config()
        config.update({
            "reduction_ratio": self.reduction_ratio,
        })
        return config


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = None

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            filters=1,
            kernel_size=self.kernel_size,
            padding="same",
            kernel_initializer="he_normal",
            use_bias=True
        )
        super().build(input_shape)

    def call(self, inputs):
        x = tf.cast(inputs, tf.float32)

        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)

        attention = tf.nn.sigmoid(self.conv(concat))
        attention = tf.cast(attention, inputs.dtype)

        return inputs * attention

    def get_config(self):
        config = super().get_config()
        config.update({
            "kernel_size": self.kernel_size,
        })
        return config


class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.kernel_size = kernel_size

        self.channel_attention = ChannelAttention(
            reduction_ratio=reduction_ratio
        )

        self.spatial_attention = SpatialAttention(
            kernel_size=kernel_size
        )

    def build(self, input_shape):
        self.channel_attention.build(input_shape)
        self.spatial_attention.build(input_shape)
        super().build(input_shape)

    def call(self, inputs):
        x = self.channel_attention(inputs)
        x = self.spatial_attention(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "reduction_ratio": self.reduction_ratio,
            "kernel_size": self.kernel_size,
        })
        return config

print("✓ CBAM Attention Modules defined")

✓ CBAM Attention Modules defined


**CUSTOM FOCAL LOSS WITH LABEL SMOOTHING**

In [5]:
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.0,
                 reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE,
                 name="focal_loss", **kwargs):
        super().__init__(reduction=reduction, name=name, **kwargs)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # Optional label smoothing
        if self.label_smoothing > 0:
            y_true = y_true * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)

        alpha_factor = y_true * self.alpha + (1.0 - y_true) * (1.0 - self.alpha)

        modulating_factor = tf.pow(1.0 - p_t, self.gamma)

        loss = -alpha_factor * modulating_factor * tf.math.log(p_t)
        return tf.squeeze(loss, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({
            "alpha": self.alpha,
            "gamma": self.gamma,
            "label_smoothing": self.label_smoothing,
        })
        return config

print("✓ Improved Focal Loss defined")

✓ Improved Focal Loss defined


**MODEL ARCHITECTURE**

In [6]:
def build_glaucoma_model(config):

    inputs = layers.Input(shape=config["input_size"], name="input_image")

    # --- Backbone ---
    backbone = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling=None
    )
    backbone.trainable = False

    x = backbone.output  # (None, 7, 7, 1280)

    # --- CBAM ---
    x = CBAM(
        reduction_ratio=config["cbam_reduction"],
        kernel_size=7,
        name="cbam"
    )(x)

    # --- Head ---
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # Block 1
    x1 = layers.Dense(512, kernel_initializer="he_normal")(x)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation("swish")(x1)
    x1 = layers.Dropout(0.4)(x1)

    # Block 2
    x2 = layers.Dense(256, kernel_initializer="he_normal")(x1)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.Activation("swish")(x2)
    x2 = layers.Dropout(0.3)(x2)

    # Residual connection
    res = layers.Dense(256, kernel_initializer="he_normal")(x)
    x = layers.Add()([x2, res])

    # Block 3
    x = layers.Dense(128, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    x = layers.Dropout(0.2)(x)

    # Output
    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="output"
    )(x)

    return Model(inputs, outputs, name="GlaucomaDetector_v3")


print("Building improved model...\n")
model = build_glaucoma_model(CONFIG)
model.summary(show_trainable=True)

Building improved model...

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "GlaucomaDetector_v3"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_image       │ (None, 224,     │         0 │ -              │   -   │
│ (InputLayer)      │ 224, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ rescaling         │ (None, 224,     │         0 │ input_image[0… │   -   │
│ (Rescaling)       │ 224, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ normalization     │ (None, 224,     │         7 │ rescaling[0][… │   N   │
│ (Normalization)   │ 224, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ rescaling_1       │ (None, 224,     │         0 │ normalization… │   -   │
│ (Rescaling)       │ 224, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_conv_pad     │ (None, 225,     │         0 │ rescaling_1[0… │   -   │
│ (ZeroPadding2D)   │ 225, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_conv         │ (None, 112,     │       864 │ stem_conv_pad… │   N   │
│ (Conv2D)          │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_bn           │ (None, 112,     │       128 │ stem_conv[0][… │   N   │
│ (BatchNormalizat… │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ stem_activation   │ (None, 112,     │         0 │ stem_bn[0][0]  │   -   │
│ (Activation)      │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_dwconv    │ (None, 112,     │       288 │ stem_activati… │   N   │
│ (DepthwiseConv2D) │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_bn        │ (None, 112,     │       128 │ block1a_dwcon… │   N   │
│ (BatchNormalizat… │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_activati… │ (None, 112,     │         0 │ block1a_bn[0]… │   -   │
│ (Activation)      │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_squee… │ (None, 32)      │         0 │ block1a_activ… │   -   │
│ (GlobalAveragePo… │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_resha… │ (None, 1, 1,    │         0 │ block1a_se_sq… │   -   │
│ (Reshape)         │ 32)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_reduce │ (None, 1, 1, 8) │       264 │ block1a_se_re… │   N   │
│ (Conv2D)          │                 │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_expand │ (None, 1, 1,    │       288 │ block1a_se_re… │   N   │
│ (Conv2D)          │ 32)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_se_excite │ (None, 112,     │         0 │ block1a_activ… │   -   │
│ (Multiply)        │ 112, 32)        │           │ block1a_se_ex… │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block1a_project_… │ (None, 112,     │       512 │ block1a_se_ex… │   N 

 Total params: 5,407,575 (20.63 MB)

 Trainable params: 1,356,212 (5.17 MB)

 Non-trainable params: 4,051,363 (15.45 MB)

**PHASE 1: FROZEN BACKBONE**

In [ ]:
train_ds_p1 = build_train_pipeline(train_ds_raw, CONFIG["phase1_batch_size"])
val_ds_p1 = build_eval_pipeline(val_ds_raw, CONFIG["phase1_batch_size"])

# Safe steps_per_epoch
steps_per_epoch = tf.data.experimental.cardinality(train_ds_p1).numpy()
if steps_per_epoch < 0:
    steps_per_epoch = 100

# --- LR SCHEDULE ---
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG["phase1_lr"],
    decay_steps=steps_per_epoch * CONFIG["phase1_epochs"]
)

# --- OPTIMIZER ---
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=CONFIG["weight_decay"],
    clipnorm=1.0
)

# --- COMPILE ---
model.compile(
    optimizer=optimizer,
    loss=FocalLoss(
        alpha=CONFIG["focal_alpha"],
        gamma=CONFIG["focal_gamma"],
        label_smoothing=CONFIG["label_smoothing"]
    ),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

# --- PATHS ---
phase1_best_path = os.path.join(CONFIG["save_dir"], f"{run_name}_p1_best.keras")
phase1_resume_path = os.path.join(CONFIG["save_dir"], f"{run_name}_p1_resume.keras")

# --- CALLBACKS ---
callbacks_p1 = [
    ModelCheckpoint(
        filepath=phase1_best_path,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(CONFIG["save_dir"], f"{run_name}_p1_log.csv")
    )
]

print("=" * 60)
print(" PHASE 1: Frozen Backbone")
print("=" * 60)

history_p1 = model.fit(
    train_ds_p1,
    validation_data=val_ds_p1,
    epochs=CONFIG["phase1_epochs"],
    callbacks=callbacks_p1,
    verbose=1,
    class_weight=CLASS_WEIGHT if USE_CLASS_WEIGHTS else None
)

# Save resume checkpoint
model.save(phase1_resume_path)

print(f"\n✓ Phase 1 complete! Saved at: {phase1_resume_path}")

 PHASE 1: Frozen Backbone
Epoch 1/12
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6206 - auc: 0.7317 - loss: 0.0949 - precision: 0.5119 - recall: 0.7752   
Epoch 1: val_auc improved from None to 0.87589, saving model to /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/models/NineMay/glaucoma_b0_9_may_glaucoma_p1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/models/NineMay/glaucoma_b0_9_may_glaucoma_p1_best.keras
139/139 ━━━━━━━━━━━━━━━━━━━━ 337s 2s/step - accuracy: 0.6777 - auc: 0.7783 - loss: 0.0815 - precision: 0.5609 - recall: 0.7688 - val_accuracy: 0.7990 - val_auc: 0.8759 - val_loss: 0.0579 - val_precision: 0.7440 - val_recall: 0.7323
Epoch 2/12
139/139 ━━━━━━━━━━━━━━━━━━━━ 0s 865ms/step - accuracy: 0.7423 - auc: 0.8295 - loss: 0.0660 - precision: 0.6524 - recall: 0.7553
Epoch 2: val_auc improved from 0.87589 to 0.89859, saving model to /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/models/NineMay/glaucoma_b0_9_may_glauco

**PHASE 2: FINE TUNING**

In [ ]:
from tensorflow.keras import layers

# Prefer the best Phase 1 checkpoint if it exists
phase1_best_path = os.path.join(CONFIG["save_dir"], f"{run_name}_p1_best.keras")
phase1_resume_path = os.path.join(CONFIG["save_dir"], f"{run_name}_p1_resume.keras")
load_path = phase1_best_path if os.path.exists(phase1_best_path) else phase1_resume_path

print(f"Loading Phase 1 model from: {load_path}")

model = keras.models.load_model(
    load_path,
    custom_objects={
        "CBAM": CBAM,
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention,
        "FocalLoss": FocalLoss
    },
    compile=False
)

print("✓ Loaded model from checkpoint")


# ---------- HELPERS ----------
def iter_all_layers(layer_or_model, seen=None):
    """Yield all unique layers recursively from a model or layer."""
    if seen is None:
        seen = set()

    if id(layer_or_model) in seen:
        return
    seen.add(id(layer_or_model))

    yield layer_or_model

    if hasattr(layer_or_model, "layers"):
        for sublayer in layer_or_model.layers:
            yield from iter_all_layers(sublayer, seen)


# ---------- FREEZE / UNFREEZE ----------
# Freeze everything first
for layer in iter_all_layers(model):
    if layer is not model:
        layer.trainable = False

# Unfreeze late EfficientNet blocks anywhere in the model
UNFREEZE_BLOCKS = ("block6", "block7", "top_conv", "top_bn", "top_activation")
unfrozen_count = 0

for layer in iter_all_layers(model):
    lname = layer.name.lower()
    if any(k in lname for k in UNFREEZE_BLOCKS):
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
            unfrozen_count += 1

# Keep the head trainable
HEAD_LAYER_HINTS = ("cbam", "gap", "dense", "dropout", "output")
for layer in iter_all_layers(model):
    lname = layer.name.lower()
    if any(k in lname for k in HEAD_LAYER_HINTS):
        layer.trainable = True

# Force all BatchNorm layers frozen
for layer in iter_all_layers(model):
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.trainable = True

print(f"✓ Unfrozen backbone layers: {unfrozen_count}")

trainable_layers = [layer.name for layer in iter_all_layers(model) if getattr(layer, "trainable", False)]
print(f"Trainable layers count: {len(trainable_layers)}")
print("Some trainable layers:")
print(trainable_layers[:30])

if len(model.trainable_weights) == 0:
    raise RuntimeError(
        "No trainable weights found. "
        "Check the layer names printed above and confirm the phase-1 model was loaded correctly."
    )


# ---------- DATA ----------
train_ds_p2 = build_train_pipeline(train_ds_raw, CONFIG["phase2_batch_size"])
val_ds_p2 = build_eval_pipeline(val_ds_raw, CONFIG["phase2_batch_size"])

steps_per_epoch = tf.data.experimental.cardinality(train_ds_p2).numpy()
if steps_per_epoch < 0:
    steps_per_epoch = 100

# ---------- LR ----------
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG["phase2_lr"],
    decay_steps=steps_per_epoch * CONFIG["phase2_epochs"]
)

optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=CONFIG["weight_decay"] * 0.5,
    clipnorm=1.0
)

# ---------- COMPILE ----------
model.compile(
    optimizer=optimizer,
    loss=FocalLoss(
        alpha=CONFIG["focal_alpha"],
        gamma=CONFIG["focal_gamma"],
        label_smoothing=CONFIG["label_smoothing"]
    ),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

# ---------- PATHS ----------
phase2_best_path = os.path.join(CONFIG["save_dir"], f"{run_name}_p2_best.keras")

# ---------- CALLBACKS ----------
callbacks_p2 = [
    ModelCheckpoint(
        filepath=phase2_best_path,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(CONFIG["save_dir"], f"{run_name}_p2_log.csv")
    )
]

print("=" * 60)
print(" PHASE 2: Fine-tuning")
print("=" * 60)

history_p2 = model.fit(
    train_ds_p2,
    validation_data=val_ds_p2,
    epochs=CONFIG["phase2_epochs"],
    callbacks=callbacks_p2,
    verbose=1,
    class_weight=CLASS_WEIGHT if USE_CLASS_WEIGHTS else None
)

print("\n✓ Phase 2 complete!")

Loading Phase 1 model from: /content/drive/MyDrive/Glumoma2.0/GlumomaDataset/models/NineMay/glaucoma_b0_9_may_glaucoma_p1_best.keras


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


✓ Loaded model from checkpoint
✓ Unfrozen backbone layers: 59
Trainable layers count: 70
Some trainable layers:
['GlaucomaDetector_v3', 'block6a_expand_conv', 'block6a_expand_activation', 'block6a_dwconv_pad', 'block6a_dwconv', 'block6a_activation', 'block6a_se_squeeze', 'block6a_se_reshape', 'block6a_se_reduce', 'block6a_se_expand', 'block6a_se_excite', 'block6a_project_conv', 'block6b_expand_conv', 'block6b_expand_activation', 'block6b_dwconv', 'block6b_activation', 'block6b_se_squeeze', 'block6b_se_reshape', 'block6b_se_reduce', 'block6b_se_expand', 'block6b_se_excite', 'block6b_project_conv', 'block6b_drop', 'block6b_add', 'block6c_expand_conv', 'block6c_expand_activation', 'block6c_dwconv', 'block6c_activation', 'block6c_se_squeeze', 'block6c_se_reshape']
 PHASE 2: Fine-tuning
Epoch 1/25
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 675ms/step - accuracy: 0.8380 - auc: 0.9233 - loss: 0.0426 - precision: 0.7709 - recall: 0.8312
Epoch 1: val_auc improved from None to 0.92539, saving model to /con